# Module 10 — Foreign Keys, Normalization, and Data Integrity

**CIS 3120 · Programming for Analytics · Baruch College / Zicklin School of Business**

---

This notebook extends the database skills you developed in Module 7.  You will work
with multi-table schemas, enforce referential integrity using foreign keys, recognize
and repair normalization problems, and write queries that traverse relationships across
tables.

**How to use this notebook**

- Read each instruction cell before running or writing any code.
- Every exercise cell contains a `# TODO` comment that marks exactly where your work goes.
- Run cells in order from top to bottom. Later exercises depend on tables created earlier.
- If you make a mistake and need to start a section over, re-run the setup cell at the
  top of that section to recreate the database.

**Sections**

1. Environment Setup  
2. Foreign Key Mechanics  
3. Normalization — Recognition and Repair  
4. Multi-Table Querying with Real Relationships  
5. Data Integrity Scenarios  
6. Schema Design Challenge  


---
## Section 1 — Environment Setup

Run the cell below to import `sqlite3` and define two helper functions you will
use throughout the notebook:

- `run(conn, sql)` — executes a SQL statement and prints any results as a formatted table.
- `show_tables(conn)` — lists all tables currently in the database.

You do not need to modify this cell.


In [ ]:
import sqlite3

def run(conn, sql, params=()):
    """Execute a SQL statement and print results, if any."""
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    rows = cur.fetchall()
    if rows:
        cols = [d[0] for d in cur.description]
        # Header
        col_widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0))
                      for i, c in enumerate(cols)]
        header = "  ".join(str(c).ljust(col_widths[i]) for i, c in enumerate(cols))
        separator = "  ".join("-" * col_widths[i] for i in range(len(cols)))
        print(header)
        print(separator)
        for row in rows:
            print("  ".join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
        print(f"\n({len(rows)} row{'s' if len(rows) != 1 else ''})")
    else:
        print("Statement executed successfully. No rows returned.")

def show_tables(conn):
    """List all tables in the database."""
    run(conn, "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")


---
## Section 2 — Foreign Key Mechanics

### Background

A **foreign key** is a column (or set of columns) in one table whose values must match
values in the primary key of another table.  This relationship is called
**referential integrity**: the database will reject any operation that would leave a
child row pointing to a parent row that does not exist.

SQLite does not enforce foreign keys by default.  You must enable enforcement at the
start of each connection with:

```sql
PRAGMA foreign_keys = ON;
```

In this section you will:

1. Create a two-table schema with a foreign key constraint.  
2. Observe what happens when you try to insert a row that violates the constraint.  
3. Correct the insert and confirm the relationship works as intended.

### The scenario

A small veterinary clinic tracks **owners** and their **pets**.  Every pet record must
reference a valid owner.

---

### Exercise 2.1 — Enable foreign key enforcement and create the schema


In [ ]:
# Setup — run this cell to create a fresh in-memory database for Section 2.
conn2 = sqlite3.connect(":memory:")

# Enable foreign key enforcement. This must be run once per connection.
conn2.execute("PRAGMA foreign_keys = ON;")
print("Foreign key enforcement is ON.")


The two tables you need are defined below.  Study the schema carefully, then run the
cell to create them.

```sql
CREATE TABLE Owner (
    owner_id   INTEGER PRIMARY KEY,
    full_name  TEXT    NOT NULL,
    phone      TEXT
);

CREATE TABLE Pet (
    pet_id     INTEGER PRIMARY KEY,
    pet_name   TEXT    NOT NULL,
    species    TEXT    NOT NULL,
    owner_id   INTEGER NOT NULL,
    FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
);
```

Notice that `Pet.owner_id` is declared as a `FOREIGN KEY` that references `Owner.owner_id`.
Any `pet_id` inserted into `Pet` must correspond to an existing row in `Owner`.


In [ ]:
conn2.execute("DROP TABLE IF EXISTS Pet;")
conn2.execute("DROP TABLE IF EXISTS Owner;")

conn2.execute("""
    CREATE TABLE Owner (
        owner_id   INTEGER PRIMARY KEY,
        full_name  TEXT    NOT NULL,
        phone      TEXT
    );
""")

conn2.execute("""
    CREATE TABLE Pet (
        pet_id     INTEGER PRIMARY KEY,
        pet_name   TEXT    NOT NULL,
        species    TEXT    NOT NULL,
        owner_id   INTEGER NOT NULL,
        FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
    );
""")

conn2.commit()
show_tables(conn2)


---

### Exercise 2.2 — rowid, AUTOINCREMENT, and SQLite system tables

SQLite assigns every row an internal integer identifier called **rowid**.  When you
declare a column as `INTEGER PRIMARY KEY`, that column becomes an alias for `rowid`
and SQLite fills it in automatically if you do not supply a value.

Adding the `AUTOINCREMENT` keyword strengthens this guarantee: SQLite records the
highest ID ever used in a system table called `sqlite_sequence` and never reuses
that value, even after a row is deleted.

A second system table, `sqlite_master`, is SQLite's internal catalog of every table,
index, view, and trigger in the database.

In this exercise you will:

1. Create a table **with** `AUTOINCREMENT` and observe `sqlite_sequence`.
2. Insert rows without supplying an ID and confirm SQLite fills it in.
3. Delete a row, then observe that the next inserted ID does **not** reuse the
   deleted value.
4. Query `sqlite_master` to inspect the schema of your own tables.


In [ ]:
# Setup — fresh connection for this exercise
conn2b = sqlite3.connect(":memory:")
conn2b.execute("PRAGMA foreign_keys = ON;")

# Part A: create a table using AUTOINCREMENT
conn2b.execute("""
    CREATE TABLE Animal (
        animal_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        common_name TEXT    NOT NULL,
        species     TEXT    NOT NULL
    );
""")
conn2b.commit()

# Part B: insert three rows WITHOUT supplying animal_id
# SQLite assigns the IDs automatically
conn2b.executemany(
    "INSERT INTO Animal (common_name, species) VALUES (?, ?);",
    [
        ("Mango",   "Canis lupus familiaris"),   # TODO: replace with your own animals if you like
        ("Shadow",  "Felis catus"),
        ("Biscuit", "Oryctolagus cuniculus"),
    ]
)
conn2b.commit()

print("=== Animal table after three inserts ===")
run(conn2b, "SELECT * FROM Animal;")

# Part C: query sqlite_sequence to see the highest-ever-used ID
print("\n=== sqlite_sequence ===")
run(conn2b, "SELECT * FROM sqlite_sequence;")

# Part D: delete the row with animal_id = 3, then insert a new row
conn2b.execute("DELETE FROM Animal WHERE animal_id = 3;")
conn2b.execute("INSERT INTO Animal (common_name, species) VALUES ('Noodle', 'Canis lupus familiaris');")
conn2b.commit()

print("\n=== Animal table after deleting ID 3 and inserting a new row ===")
run(conn2b, "SELECT * FROM Animal;")
print()
print("Notice: the new row received ID 4, not 3.")
print("AUTOINCREMENT prevents reuse of the deleted value.")

# Part E: query sqlite_sequence again — seq should now be 4
print("\n=== sqlite_sequence after the fourth insert ===")
run(conn2b, "SELECT * FROM sqlite_sequence;")

# Part F: inspect sqlite_master to see the CREATE TABLE statement SQLite stored
print("\n=== sqlite_master — schema catalog ===")
run(conn2b, "SELECT name, sql FROM sqlite_master WHERE type = 'table';")


**Reflection:**

> 1. The fourth animal received ID 4. It did not receive ID 3 because `AUTOINCREMENT` records the highest ID ever assigned (3) in `sqlite_sequence` and guarantees the next ID is always strictly greater than that maximum, even after the row with ID 3 was deleted.

> 2. The `seq` column in `sqlite_sequence` holds the largest `rowid` ever inserted into that table. SQLite updates it each time a new row is inserted, to ensure future inserts never reuse an old value.

> 3. The `sql` column in `sqlite_master` contains the original `CREATE TABLE` (or `CREATE INDEX`, etc.) SQL statement that was used to define the object, exactly as it was written.

> 4. No, `sqlite_sequence` would not exist if `Animal` used `INTEGER PRIMARY KEY` without `AUTOINCREMENT`. SQLite only creates and maintains the `sqlite_sequence` table when at least one table is declared with `AUTOINCREMENT`; without it, SQLite simply picks `max(rowid) + 1` on each insert without recording any history.


---

### Exercise 2.3 — Insert valid owner rows

Insert the following three owners into the `Owner` table.  Use a single `executemany`
call with a list of tuples.

| owner_id | full_name       | phone        |
|:---------|:----------------|:-------------|
| 1        | Maria Vasquez   | 718-555-0101 |
| 2        | James O'Brien   | 212-555-0188 |
| 3        | Priya Nair      | 347-555-0234 |


In [ ]:
owners = [
    (1, "Maria Vasquez", "718-555-0101"),
    (2, "James O'Brien",  "212-555-0188"),
    (3, "Priya Nair",     "347-555-0234"),
]

conn2.executemany(
    "INSERT INTO Owner (owner_id, full_name, phone) VALUES (?, ?, ?);",
    owners
)

conn2.commit()
run(conn2, "SELECT * FROM Owner;")


---

### Exercise 2.4 — Attempt an invalid insert

Try to insert a pet whose `owner_id` does not exist in the `Owner` table.  The cell
below wraps the insert in a `try/except` block so the notebook can continue after the
error.

Complete the `INSERT` statement with `owner_id = 99` (which does not exist).  Read the
error message that SQLite raises and write a one-sentence explanation in the markdown
cell that follows.


In [ ]:
invalid_insert = """
    INSERT INTO Pet (pet_id, pet_name, species, owner_id)
    VALUES (99, 'Ghost', 'Cat', 99)
"""

try:
    conn2.execute(invalid_insert)
    conn2.commit()
    print("Insert succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError caught: {e}")


**Your explanation:**

> SQLite raised an `IntegrityError` because the `owner_id` value 99 does not exist in the `Owner` table, violating the `FOREIGN KEY` constraint that requires every `Pet.owner_id` to reference a valid row in `Owner`.


---

### Exercise 2.5 — Insert valid pet rows

Now insert the following pets using valid `owner_id` values.

| pet_id | pet_name | species | owner_id |
|:-------|:---------|:--------|:---------|
| 1      | Mango    | Dog     | 1        |
| 2      | Shadow   | Cat     | 1        |
| 3      | Biscuit  | Rabbit  | 2        |
| 4      | Noodle   | Dog     | 3        |


In [ ]:
pets = [
    (1, "Mango",   "Dog",    1),
    (2, "Shadow",  "Cat",    1),
    (3, "Biscuit", "Rabbit", 2),
    (4, "Noodle",  "Dog",    3),
]

conn2.executemany(
    "INSERT INTO Pet (pet_id, pet_name, species, owner_id) VALUES (?, ?, ?, ?);",
    pets
)

conn2.commit()
run(conn2, "SELECT * FROM Pet;")


---

### Exercise 2.6 — Verify enforcement: delete a parent row

Try to delete owner 1 (Maria Vasquez), who has two pets in the `Pet` table.

Predict what will happen before you run the cell, then observe the result.


In [ ]:
try:
    conn2.execute("DELETE FROM Owner WHERE owner_id = 1;")
    conn2.commit()
    print("Delete succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError caught: {e}")


**Reflection:**

> The delete failed because owner 1 (Maria Vasquez) still has child rows in the `Pet` table that reference her `owner_id`, and deleting the parent would leave those rows pointing to a non-existent owner, violating referential integrity. To delete owner 1 along with her pets, you must first delete the pet rows where `owner_id = 1`, and then delete the owner row.


---
## Section 3 — Normalization — Recognition and Repair

### Background

**Normalization** is the process of organizing a database schema to reduce redundancy
and prevent update anomalies.  The three most commonly applied normal forms are:

| Normal Form | Key Rule |
|:------------|:---------|
| **1NF** | Every column holds a single, atomic value.  No repeating groups. |
| **2NF** | 1NF plus: every non-key column depends on the *whole* primary key (matters when the key is composite). |
| **3NF** | 2NF plus: no non-key column depends on another non-key column (no transitive dependencies). |

A **denormalized** table violates one or more of these rules.  Common symptoms include:

- The same value repeated across many rows (e.g., a customer's address on every order row).
- Columns that describe something other than the entity the table is meant to track.
- Difficulty updating one piece of information without touching many rows.

---

### Exercise 3.1 — Identify the normalization problem

The table below represents a course enrollment system stored as a single flat table.

```
EnrollmentFlat
-----------------------------------------------------------------------
student_id | student_name | student_email       | course_id | course_title         | credits | instructor
-----------|--------------|---------------------|-----------|----------------------|---------|----------
1          | Ana Torres   | atorres@bcc.cuny.edu | CIS3120   | Prog for Analytics   | 3       | Prof. Kim
1          | Ana Torres   | atorres@bcc.cuny.edu | CIS2200   | Business Data Comm   | 3       | Prof. Lau
2          | Ben Reyes    | breyes@bcc.cuny.edu  | CIS3120   | Prog for Analytics   | 3       | Prof. Kim
3          | Cleo Marsh   | cmarsh@bcc.cuny.edu  | CIS4800   | Database Management  | 3       | Prof. Osei
```

In the markdown cell below, answer the following questions:

1. What data is repeated across multiple rows?
2. What anomaly would occur if Prof. Kim's name changed?
3. What normal form does this table violate, and why?


**Your answers:**

> 1. Student information (`student_id`, `student_name`, `student_email`) is repeated on every row for each course a student is enrolled in, and course information (`course_id`, `course_title`, `credits`, `instructor`) is repeated for every student in that course.

> 2. If Prof. Kim's name changed, every row in the table that references CIS3120 would need to be updated individually; missing even one row would leave the data in an inconsistent state — an update anomaly.

> 3. This table violates **Third Normal Form (3NF)** (and also 2NF when considering a composite key of `student_id` + `course_id`): columns like `student_name` depend only on `student_id` (not the full composite key), and `course_title`, `credits`, and `instructor` depend only on `course_id` — these are partial and transitive dependencies that 3NF prohibits.


---

### Exercise 3.2 — Decompose the flat table into a normalized schema

The flat `EnrollmentFlat` table should be split into three tables:

- `Student (student_id, student_name, student_email)`
- `Course (course_id, course_title, credits, instructor)`
- `Enrollment (student_id, course_id)` — a junction table linking students to courses

The setup cell below creates `EnrollmentFlat` and populates it.  Your task is to:

1. Create the three normalized tables with appropriate primary and foreign keys.
2. Populate each table by selecting data from `EnrollmentFlat`.


In [ ]:
# Setup — creates EnrollmentFlat. Run this cell first.
conn3 = sqlite3.connect(":memory:")
conn3.execute("PRAGMA foreign_keys = ON;")

conn3.execute("""
    CREATE TABLE EnrollmentFlat (
        student_id     INTEGER,
        student_name   TEXT,
        student_email  TEXT,
        course_id      TEXT,
        course_title   TEXT,
        credits        INTEGER,
        instructor     TEXT
    );
""")

conn3.executemany(
    "INSERT INTO EnrollmentFlat VALUES (?, ?, ?, ?, ?, ?, ?);",
    [
        (1, "Ana Torres",  "atorres@bcc.cuny.edu", "CIS3120", "Prog for Analytics",  3, "Prof. Kim"),
        (1, "Ana Torres",  "atorres@bcc.cuny.edu", "CIS2200", "Business Data Comm",  3, "Prof. Lau"),
        (2, "Ben Reyes",   "breyes@bcc.cuny.edu",  "CIS3120", "Prog for Analytics",  3, "Prof. Kim"),
        (3, "Cleo Marsh",  "cmarsh@bcc.cuny.edu",  "CIS4800", "Database Management", 3, "Prof. Osei"),
    ]
)
conn3.commit()
print("EnrollmentFlat loaded.")
run(conn3, "SELECT * FROM EnrollmentFlat;")


In [ ]:
# Create the Student table
conn3.execute("""
    CREATE TABLE Student (
        student_id    INTEGER PRIMARY KEY,
        student_name  TEXT    NOT NULL,
        student_email TEXT
    );
""")

# Create the Course table
conn3.execute("""
    CREATE TABLE Course (
        course_id    TEXT PRIMARY KEY,
        course_title TEXT    NOT NULL,
        credits      INTEGER,
        instructor   TEXT
    );
""")

# Create the Enrollment junction table
conn3.execute("""
    CREATE TABLE Enrollment (
        student_id INTEGER,
        course_id  TEXT,
        PRIMARY KEY (student_id, course_id),
        FOREIGN KEY (student_id) REFERENCES Student(student_id),
        FOREIGN KEY (course_id)  REFERENCES Course(course_id)
    );
""")

conn3.commit()
show_tables(conn3)


In [ ]:
# Populate Student using INSERT INTO ... SELECT DISTINCT from EnrollmentFlat
conn3.execute("""
    INSERT INTO Student (student_id, student_name, student_email)
    SELECT DISTINCT student_id, student_name, student_email
    FROM EnrollmentFlat;
""")

# Populate Course using INSERT INTO ... SELECT DISTINCT from EnrollmentFlat
conn3.execute("""
    INSERT INTO Course (course_id, course_title, credits, instructor)
    SELECT DISTINCT course_id, course_title, credits, instructor
    FROM EnrollmentFlat;
""")

# Populate Enrollment using INSERT INTO ... SELECT from EnrollmentFlat
conn3.execute("""
    INSERT INTO Enrollment (student_id, course_id)
    SELECT student_id, course_id
    FROM EnrollmentFlat;
""")

conn3.commit()

print("=== Student ===")
run(conn3, "SELECT * FROM Student;")
print("\n=== Course ===")
run(conn3, "SELECT * FROM Course;")
print("\n=== Enrollment ===")
run(conn3, "SELECT * FROM Enrollment;")


---
## Section 4 — Multi-Table Querying with Real Relationships

### Background

Once data is spread across related tables, answering most business questions requires
combining rows from two or more tables.

- An **INNER JOIN** (or simply `JOIN`) returns only rows where the join condition is
  satisfied in *both* tables.
- A **LEFT JOIN** returns all rows from the left table and matching rows from the right
  table.  Where there is no match, the right-side columns appear as `NULL`.  This is
  useful for finding records with *no* matching child rows.

In this section you will query the library database introduced in Module 7 — expanded
here to include a `Fine` table.

---

### Exercise 4.1 — Setup: the expanded library database


In [ ]:
# Setup — run this cell to create the library database.
conn4 = sqlite3.connect(":memory:")
conn4.execute("PRAGMA foreign_keys = ON;")

conn4.executescript("""
    CREATE TABLE Member (
        member_id    INTEGER PRIMARY KEY,
        full_name    TEXT    NOT NULL,
        join_date    TEXT    NOT NULL
    );

    CREATE TABLE Book (
        book_id      INTEGER PRIMARY KEY,
        title        TEXT    NOT NULL,
        author       TEXT    NOT NULL,
        genre        TEXT
    );

    CREATE TABLE Loan (
        loan_id      INTEGER PRIMARY KEY,
        member_id    INTEGER NOT NULL,
        book_id      INTEGER NOT NULL,
        loan_date    TEXT    NOT NULL,
        return_date  TEXT,
        FOREIGN KEY (member_id) REFERENCES Member(member_id),
        FOREIGN KEY (book_id)   REFERENCES Book(book_id)
    );

    CREATE TABLE Fine (
        fine_id      INTEGER PRIMARY KEY,
        loan_id      INTEGER NOT NULL UNIQUE,
        amount       REAL    NOT NULL,
        paid         INTEGER NOT NULL DEFAULT 0,
        FOREIGN KEY (loan_id) REFERENCES Loan(loan_id)
    );
""")

conn4.executemany("INSERT INTO Member VALUES (?, ?, ?);", [
    (1, "Amara Osei",     "2019-03-15"),
    (2, "Luis Pereira",   "2021-07-22"),
    (3, "Yuki Tanaka",    "2020-11-01"),
    (4, "Grace Nwosu",    "2018-06-10"),
])

conn4.executemany("INSERT INTO Book VALUES (?, ?, ?, ?);", [
    (1,  "Invisible Man",          "Ralph Ellison",    "Fiction"),
    (2,  "Freakonomics",           "Levitt & Dubner",  "Non-Fiction"),
    (3,  "The Alchemist",          "Paulo Coelho",     "Fiction"),
    (4,  "Thinking, Fast and Slow","Daniel Kahneman",  "Non-Fiction"),
    (5,  "Americanah",             "Chimamanda Adichie","Fiction"),
])

conn4.executemany("INSERT INTO Loan VALUES (?, ?, ?, ?, ?);", [
    (1, 1, 1, "2023-01-10", "2023-01-28"),
    (2, 1, 3, "2023-02-05", "2023-03-01"),
    (3, 2, 2, "2023-01-20", "2023-02-10"),
    (4, 3, 4, "2023-03-01", None),
    (5, 4, 5, "2022-12-01", "2023-01-15"),
    (6, 4, 1, "2023-04-10", None),
])

conn4.executemany("INSERT INTO Fine VALUES (?, ?, ?, ?);", [
    (1, 2, 3.50, 0),
    (2, 5, 7.00, 1),
])

conn4.commit()
print("Library database loaded.")
show_tables(conn4)


---

### Exercise 4.2 — List all loans with member name and book title

Write a query that returns each loan with the borrowing member's name and the book
title.  Include: `loan_id`, `full_name`, `title`, `loan_date`, and `return_date`.
Order the results by `loan_date`.


In [ ]:
query_4_2 = """
    SELECT l.loan_id,
           m.full_name,
           b.title,
           l.loan_date,
           l.return_date
    FROM   Loan   l
    JOIN   Member m ON l.member_id = m.member_id
    JOIN   Book   b ON l.book_id   = b.book_id
    ORDER  BY l.loan_date;
"""

run(conn4, query_4_2)


---

### Exercise 4.3 — Find members who have never borrowed a book

Use a `LEFT JOIN` to identify members who have no rows in the `Loan` table.

Hint: after a `LEFT JOIN`, a member with no loans will have `NULL` in any
`Loan` column.  Use `WHERE loan_id IS NULL` to filter for those rows.


In [ ]:
query_4_3 = """
    SELECT m.member_id,
           m.full_name
    FROM   Member m
    LEFT JOIN Loan l ON m.member_id = l.member_id
    WHERE  l.loan_id IS NULL;
"""

run(conn4, query_4_3)


---

### Exercise 4.4 — List all outstanding loans with any associated fine amount

Write a query that shows loans where `return_date IS NULL`.  For each such loan,
include the member name, book title, loan date, and the fine amount if one exists
(show `NULL` if no fine has been issued).

Hint: use an `INNER JOIN` to reach `Member` and `Book`, and a `LEFT JOIN` to
reach `Fine` (since not every outstanding loan has a fine).


In [ ]:
query_4_4 = """
    SELECT l.loan_id,
           m.full_name,
           b.title,
           l.loan_date,
           f.amount AS fine_amount
    FROM   Loan   l
    JOIN   Member m ON l.member_id = m.member_id
    JOIN   Book   b ON l.book_id   = b.book_id
    LEFT JOIN Fine f ON l.loan_id  = f.loan_id
    WHERE  l.return_date IS NULL;
"""

run(conn4, query_4_4)


---

### Exercise 4.5 — Aggregate: total fines owed per member

Write a query that returns each member's name and the total unpaid fine amount
they owe.  Only include members who have at least one unpaid fine (`paid = 0`).

Use `JOIN`, `GROUP BY`, and the `SUM()` aggregate function.


In [ ]:
query_4_5 = """
    SELECT m.full_name,
           SUM(f.amount) AS total_unpaid
    FROM   Fine   f
    JOIN   Loan   l ON f.loan_id   = l.loan_id
    JOIN   Member m ON l.member_id = m.member_id
    WHERE  f.paid = 0
    GROUP  BY m.member_id, m.full_name;
"""

run(conn4, query_4_5)


---
## Section 5 — Data Integrity Scenarios

### Background

Foreign key constraints protect data integrity but also impose an ordering requirement
on operations:

- You must **insert parent rows before child rows**.
- You must **delete child rows before parent rows** (or use `ON DELETE CASCADE`).
- Updating a primary key value that is referenced by a foreign key requires similar care.

This section gives you practice recognizing and resolving the most common integrity
errors.

---

### Exercise 5.1 — Correct deletion order

The cell below attempts to delete book 1 (*Invisible Man*) directly from the `Book`
table.  This will fail because `Loan` rows reference it.

1. Run the cell and read the error.
2. In the second cell, write a corrected deletion sequence that removes all loans for
   book 1 first, then deletes the book itself.

Note: you are working on `conn4` from Section 4, which already has data loaded.


In [ ]:
try:
    conn4.execute("DELETE FROM Book WHERE book_id = 1;")
    conn4.commit()
    print("Delete succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
    conn4.rollback()


In [ ]:
# First, delete any Fine rows linked to loans for book_id = 1
conn4.execute("""
    DELETE FROM Fine
    WHERE  loan_id IN (SELECT loan_id FROM Loan WHERE book_id = 1)
""")

# Next, delete the Loan rows for book_id = 1
conn4.execute("""
    DELETE FROM Loan WHERE book_id = 1
""")

# Finally, delete the Book row
conn4.execute("""
    DELETE FROM Book WHERE book_id = 1
""")

conn4.commit()
print("Deletion sequence completed.")
run(conn4, "SELECT * FROM Book;")


---

### Exercise 5.2 — PRAGMA foreign_keys OFF vs ON

This exercise demonstrates why the PRAGMA matters.

The cell below connects to a *new* in-memory database **without** enabling foreign keys.
It creates the same `Owner` / `Pet` schema from Section 2 but inserts a pet with a
non-existent `owner_id`.  Run it and observe that no error is raised.

Then modify the cell to add `conn5.execute("PRAGMA foreign_keys = ON;")` immediately
after the connection is opened, re-run, and observe the difference.


In [ ]:
conn5 = sqlite3.connect(":memory:")
conn5.execute("PRAGMA foreign_keys = ON;")  # Added: enables FK enforcement

conn5.execute("""
    CREATE TABLE Owner (
        owner_id  INTEGER PRIMARY KEY,
        full_name TEXT NOT NULL
    );
""")
conn5.execute("""
    CREATE TABLE Pet (
        pet_id    INTEGER PRIMARY KEY,
        pet_name  TEXT    NOT NULL,
        owner_id  INTEGER NOT NULL,
        FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
    );
""")
conn5.commit()

try:
    conn5.execute("INSERT INTO Pet VALUES (1, 'Ghost', 999);")
    conn5.commit()
    print("Insert succeeded — foreign key was NOT enforced.")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e} — foreign key WAS enforced.")


**Reflection:**

> SQLite defaults to foreign key enforcement being OFF for backwards compatibility: older applications and SQLite versions did not support foreign keys at all, so the default-off setting ensures they continue to run without unexpected errors. For Python programs using `sqlite3`, this means that foreign key constraints are silently ignored unless `PRAGMA foreign_keys = ON` is explicitly executed on every new connection before any INSERT, UPDATE, or DELETE operations are performed.


---
## Section 6 — Schema Design Challenge

### The scenario

A community gym wants to track its **members**, the **fitness classes** it offers, and
member **enrollments** in those classes.  The following rules apply:

- Each member has an ID, a full name, a membership start date, and a membership type
  (`"Basic"` or `"Premium"`).
- Each fitness class has an ID, a class name, an instructor name, a day of the week,
  and a maximum capacity (integer).
- A member can enroll in many classes; a class can have many members (many-to-many).
- An enrollment records which member joined which class and the date they enrolled.
- A member may not enroll in the same class more than once.

---

### Exercise 6.1 — Design the schema

In the markdown cell below, list the three tables you will create, their columns, and
identify the primary key and any foreign keys for each table.  Use plain text — no SQL
yet.


**Your schema design:**

> **Member table**
> - `member_id` INTEGER — **Primary Key**
> - `full_name` TEXT NOT NULL
> - `start_date` TEXT NOT NULL
> - `membership_type` TEXT NOT NULL  (values: 'Basic' or 'Premium')

> **FitnessClass table**
> - `class_id` INTEGER — **Primary Key**
> - `class_name` TEXT NOT NULL
> - `instructor` TEXT NOT NULL
> - `day_of_week` TEXT NOT NULL
> - `max_capacity` INTEGER NOT NULL

> **Enrollment table**
> - `member_id` INTEGER — **Foreign Key** → Member(member_id), part of composite PK
> - `class_id` INTEGER — **Foreign Key** → FitnessClass(class_id), part of composite PK
> - `enroll_date` TEXT NOT NULL
> - **Composite Primary Key**: (member_id, class_id) — prevents a member from enrolling in the same class more than once


---

### Exercise 6.2 — Implement the schema

Write and execute the three `CREATE TABLE` statements.  Remember to:

- Enable `PRAGMA foreign_keys = ON`.
- Use a composite primary key on the `Enrollment` table to prevent duplicate enrollments.
- Declare `FOREIGN KEY` constraints on `Enrollment`.


In [ ]:
conn6 = sqlite3.connect(":memory:")
conn6.execute("PRAGMA foreign_keys = ON;")

# CREATE TABLE Member
conn6.execute("""
    CREATE TABLE Member (
        member_id       INTEGER PRIMARY KEY,
        full_name       TEXT    NOT NULL,
        start_date      TEXT    NOT NULL,
        membership_type TEXT    NOT NULL
    );
""")

# CREATE TABLE FitnessClass
conn6.execute("""
    CREATE TABLE FitnessClass (
        class_id     INTEGER PRIMARY KEY,
        class_name   TEXT    NOT NULL,
        instructor   TEXT    NOT NULL,
        day_of_week  TEXT    NOT NULL,
        max_capacity INTEGER NOT NULL
    );
""")

# CREATE TABLE Enrollment with composite PK and foreign keys
conn6.execute("""
    CREATE TABLE Enrollment (
        member_id   INTEGER NOT NULL,
        class_id    INTEGER NOT NULL,
        enroll_date TEXT    NOT NULL,
        PRIMARY KEY (member_id, class_id),
        FOREIGN KEY (member_id) REFERENCES Member(member_id),
        FOREIGN KEY (class_id)  REFERENCES FitnessClass(class_id)
    );
""")

conn6.commit()
show_tables(conn6)


---

### Exercise 6.3 — Populate the database

Insert at least three members, three fitness classes, and five enrollments of your
choice.  At least one member should be enrolled in more than one class.


In [ ]:
# Insert 3 members
conn6.executemany(
    "INSERT INTO Member VALUES (?, ?, ?, ?);",
    [
        (1, "Aisha Kamara",   "2023-01-15", "Premium"),
        (2, "Derek Huang",    "2023-03-01", "Basic"),
        (3, "Sofia Mendez",   "2022-11-20", "Premium"),
    ]
)

# Insert 3 fitness classes
conn6.executemany(
    "INSERT INTO FitnessClass VALUES (?, ?, ?, ?, ?);",
    [
        (1, "Yoga Flow",     "Coach Rivera", "Monday",    15),
        (2, "HIIT Blast",    "Coach Patel",  "Wednesday", 20),
        (3, "Spin Cycle",    "Coach Torres", "Friday",    12),
    ]
)

# Insert 5 enrollments (Aisha in two classes)
conn6.executemany(
    "INSERT INTO Enrollment VALUES (?, ?, ?);",
    [
        (1, 1, "2023-01-20"),   # Aisha -> Yoga Flow
        (1, 2, "2023-01-20"),   # Aisha -> HIIT Blast
        (2, 2, "2023-03-05"),   # Derek -> HIIT Blast
        (3, 1, "2022-11-25"),   # Sofia -> Yoga Flow
        (3, 3, "2022-11-25"),   # Sofia -> Spin Cycle
    ]
)

conn6.commit()
print("=== Member ===")
run(conn6, "SELECT * FROM Member;")
print("\n=== FitnessClass ===")
run(conn6, "SELECT * FROM FitnessClass;")
print("\n=== Enrollment ===")
run(conn6, "SELECT * FROM Enrollment;")


---

### Exercise 6.4 — Query your database

Write queries to answer the following questions about your gym data.

**Query A**: List each member's full name alongside the names of the classes they
are enrolled in.


In [ ]:
query_6_a = """
    SELECT m.full_name,
           fc.class_name
    FROM   Member      m
    JOIN   Enrollment  e  ON m.member_id  = e.member_id
    JOIN   FitnessClass fc ON e.class_id   = fc.class_id
    ORDER  BY m.full_name, fc.class_name;
"""

run(conn6, query_6_a)


**Query B**: Find any fitness classes that currently have *no* enrollments.


In [ ]:
query_6_b = """
    SELECT fc.class_id,
           fc.class_name
    FROM   FitnessClass fc
    LEFT JOIN Enrollment e ON fc.class_id = e.class_id
    WHERE  e.member_id IS NULL;
"""

run(conn6, query_6_b)


**Query C**: For each class, show the class name and the count of enrolled members.
Order the results from most to least enrolled.


In [ ]:
query_6_c = """
    SELECT fc.class_name,
           COUNT(e.member_id) AS enrolled_count
    FROM   FitnessClass fc
    JOIN   Enrollment   e  ON fc.class_id = e.class_id
    GROUP  BY fc.class_id, fc.class_name
    ORDER  BY enrolled_count DESC;
"""

run(conn6, query_6_c)


---
## Notebook Complete

You have worked through:

- Enabling and testing foreign key enforcement in SQLite
- Inserting and deleting rows while respecting referential integrity
- Identifying normalization problems in a flat table and decomposing it into a
  properly related schema
- Writing `INNER JOIN` and `LEFT JOIN` queries that traverse foreign key relationships
- Designing and implementing a normalized three-table schema from a written specification

**Submission**: Push this completed notebook to your GitHub repository and submit the
link on Brightspace.
